# 🧠 MEMS ML System - LSTM Training Notebook

Train an LSTM model for **Remaining Useful Life (RUL) prediction** using Google Colab's free GPU.

## Quick Start:
1. Go to **Runtime → Change runtime type → GPU (T4)**
2. Configure your training parameters below ⬇️
3. Click **Runtime → Run all**

---
# ⚙️ Configuration - Edit These Values!

In [ ]:
#@title 🎛️ Training Configuration { display-mode: "form" }

#@markdown ### Training Parameters
EPOCHS = 50  #@param {type:"slider", min:5, max:100, step:5}
BATCH_SIZE = 32  #@param [16, 32, 64, 128] {type:"raw"}
EARLY_STOPPING_PATIENCE = 10  #@param {type:"slider", min:3, max:20, step:1}

#@markdown ### Data Parameters
TRAINING_SAMPLES = 5000  #@param {type:"slider", min:1000, max:10000, step:1000}
SEQUENCE_LENGTH = 30  #@param {type:"slider", min:10, max:50, step:5}

print("✅ Configuration Saved!")
print(f"   📊 Epochs: {EPOCHS} (with early stopping patience: {EARLY_STOPPING_PATIENCE})")
print(f"   📦 Batch Size: {BATCH_SIZE}")
print(f"   🔢 Training Samples: {TRAINING_SAMPLES}")
print(f"   📏 Sequence Length: {SEQUENCE_LENGTH}")

In [ ]:
# Install required packages
!pip install tensorflow numpy PyGithub -q

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import json
import base64

# Check GPU
print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Generate synthetic MEMS sensor data for training
def generate_synthetic_rul_data(n_samples=5000, sequence_length=30):
    """Generate synthetic sensor data with RUL labels"""
    np.random.seed(42)
    
    # Generate sensor readings
    time = np.linspace(0, 100, n_samples)
    
    # Simulate sensor degradation
    base_signal = np.sin(2 * np.pi * 0.1 * time) * 9.81
    noise = np.random.normal(0, 0.5, n_samples)
    degradation = np.linspace(0, 2, n_samples)  # Increasing degradation
    drift = 0.01 * time
    
    signal = base_signal + noise * (1 + degradation) + drift
    
    # Create RUL labels (100% to 0%)
    rul = np.linspace(100, 0, n_samples)
    
    # Create sequences
    X, y = [], []
    for i in range(len(signal) - sequence_length):
        X.append(signal[i:i+sequence_length])
        y.append(rul[i+sequence_length])
    
    X = np.array(X).reshape(-1, sequence_length, 1)
    y = np.array(y)
    
    return X, y

# Generate data using configured parameters
X, y = generate_synthetic_rul_data(TRAINING_SAMPLES, SEQUENCE_LENGTH)
print(f"Training data shape: X={X.shape}, y={y.shape}")

In [ ]:
# Split data
split_idx = int(len(X) * 0.8)
X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")

In [ ]:
# Build LSTM Model
def build_lstm_model(sequence_length=30, n_features=1):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(sequence_length, n_features)),
        BatchNormalization(),
        Dropout(0.2),
        
        LSTM(32, return_sequences=False),
        BatchNormalization(),
        Dropout(0.2),
        
        Dense(16, activation='relu'),
        Dense(1, activation='linear')  # RUL prediction
    ])
    
    model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['mae']
    )
    
    return model

# Create model using configured sequence length
model = build_lstm_model(SEQUENCE_LENGTH)
model.summary()

In [ ]:
# Train the model with configured parameters
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=EARLY_STOPPING_PATIENCE,
    restore_best_weights=True
)

print(f"🚀 Starting training with GPU...")
print(f"   Epochs: {EPOCHS}, Batch Size: {BATCH_SIZE}")

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

print(f"\n✅ Training complete!")
print(f"   Actual Epochs: {len(history.history['loss'])}")
print(f"   Final Loss: {history.history['loss'][-1]:.4f}")
print(f"   Final MAE: {history.history['mae'][-1]:.4f}")

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss plot
axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_title('Model Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

# MAE plot
axes[1].plot(history.history['mae'], label='Training MAE')
axes[1].plot(history.history['val_mae'], label='Validation MAE')
axes[1].set_title('Model MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

In [ ]:
# Save the trained model
model.save('lstm_rul_model.keras')
print("✅ Model saved as 'lstm_rul_model.keras'")

# Save training history for the frontend chart
training_history = {
    'loss': [float(x) for x in history.history['loss']],
    'val_loss': [float(x) for x in history.history['val_loss']],
    'mae': [float(x) for x in history.history['mae']],
    'val_mae': [float(x) for x in history.history['val_mae']],
    'epochs': len(history.history['loss']),
    'final_loss': float(history.history['loss'][-1]),
    'final_mae': float(history.history['mae'][-1]),
    'trained_on': 'Google Colab with GPU',
    'training_samples': TRAINING_SAMPLES,
    'config': {
        'max_epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'sequence_length': SEQUENCE_LENGTH
    }
}

with open('training_history.json', 'w') as f:
    json.dump(training_history, f, indent=2)

print("✅ Training history saved as 'training_history.json'")

---
# 🚀 Auto-Upload to GitHub

Run the cells below to automatically upload your trained model to GitHub!

You'll need a **GitHub Personal Access Token**. Get one here:
1. Go to https://github.com/settings/tokens
2. Click "Generate new token (classic)"
3. Give it `repo` permissions
4. Copy the token

In [ ]:
# 🔐 Enter your GitHub Personal Access Token
from getpass import getpass

GITHUB_TOKEN = getpass("Enter your GitHub Personal Access Token: ")
REPO_NAME = "consixdent10/mems-ml-system"

print("✅ Token received (hidden for security)")

In [ ]:
# 📤 Auto-Upload to GitHub
from github import Github
import base64

def upload_file_to_github(g, repo_name, file_path, github_path, commit_message):
    """Upload a file to GitHub repository"""
    repo = g.get_repo(repo_name)
    
    # Read file content
    with open(file_path, 'rb') as f:
        content = f.read()
    
    # Check if file exists
    try:
        existing_file = repo.get_contents(github_path)
        # Update existing file
        repo.update_file(
            github_path,
            commit_message,
            content,
            existing_file.sha
        )
        print(f"✅ Updated: {github_path}")
    except:
        # Create new file
        repo.create_file(
            github_path,
            commit_message,
            content
        )
        print(f"✅ Created: {github_path}")

# Connect to GitHub
g = Github(GITHUB_TOKEN)

print("\n🚀 Uploading files to GitHub...\n")

# Upload training history (JSON)
upload_file_to_github(
    g, REPO_NAME,
    'training_history.json',
    'backend/saved_models/training_history.json',
    f'🤖 Auto-update: LSTM trained with {len(history.history["loss"])} epochs'
)

# Upload training plot (PNG)
upload_file_to_github(
    g, REPO_NAME,
    'training_history.png',
    'backend/saved_models/training_history.png',
    '🤖 Auto-update: LSTM training plot from Colab'
)

# Upload model (keras)
upload_file_to_github(
    g, REPO_NAME,
    'lstm_rul_model.keras',
    'backend/saved_models/lstm_rul_model.keras',
    '🤖 Auto-update: LSTM model from Colab GPU training'
)

print("\n" + "="*50)
print("🎉 All files uploaded to GitHub successfully!")
print("="*50)
print(f"\n📍 Files uploaded to:")
print(f"   https://github.com/{REPO_NAME}/tree/main/backend/saved_models")
print(f"\n⏳ Render will auto-redeploy in ~3 minutes")
print(f"🌐 Check your site: https://mems-ml-system.vercel.app")

---
## ✅ All Done!

Your trained LSTM model has been:
1. ✅ Trained on GPU with your configured parameters
2. ✅ Saved locally
3. ✅ Uploaded to GitHub automatically
4. ⏳ Render will redeploy with new model

**Visit your site in 3-5 minutes to see the updated model!**